# 04 — Base de endereços da Prefeitura (SIRF + lotes)

**Objetivo:** montar a base oficial de endereços espacializados da Prefeitura (`ref_pref`), cruzando o SIRF com o ponto de cada lote pela inscrição imobiliária.

**Entradas:** `df_sirf` e `df_lotes` (03).

**Saídas:** `03_processamento/04_base_enderecos_prefeitura/`: `ref_pref` (CEP, via, número, lat/lon) e `sirf_vias` (CEP e via de todo o SIRF, já limpos, usados no 05 para achar os CEPs não confiáveis).

**Como rodar:** de cima para baixo.

**Seções:** 1. Base de endereços da Prefeitura · 2. Gravação e validação · 3. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("04_base_enderecos_prefeitura", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# Funções e constantes do projeto usadas pelas seções (src/enderecos.py, src/cascata.py e src/geo.py).
from src.enderecos import (
    expandir_tipo_sirf, limpar_via, normalizar_numero, normalizar_texto, tirar_numero_do_fim,
)

# ===== PARÂMETROS (config.yaml) =====
COL_SIRF_INSCRICAO = cfg.parametro("sirf.col_inscricao")
COL_SIRF_LOGRADOURO = cfg.parametro("sirf.col_logradouro")   # o tipo da via (R, AV, PCA...) vem abreviado no início
COL_SIRF_NUMERO = cfg.parametro("sirf.col_numero")
COL_SIRF_CEP = cfg.parametro("sirf.col_cep")

# ===== ENTRADAS (03) =====
df_sirf = fontes.ler_processamento(cfg, "df_sirf")
df_lotes = fontes.ler_processamento(cfg, "df_lotes")

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Base de endereços da Prefeitura (SIRF + lotes)
Cruza o SIRF com os lotes georreferenciados pela inscrição imobiliária: primeiro pela inscrição igual e, para quem não encontrar, pela "área maior" (o mesmo código com o último bloco trocado por `0000`).
Monta `ref_pref`, a base de endereços da Prefeitura, com CEP, via limpa, número e coordenadas, uma linha por registro do SIRF que tem ponto.
Registros sem número entram do mesmo jeito: servem para o centro da via e para saber quais CEPs cada via tem.
Não altera `df_sirf` nem `df_lotes`, e não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 1: montando a base de endereços da Prefeitura...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_b11 = time.time()

# ===================================================================
# ETAPA 1: INSCRIÇÃO E ÁREA MAIOR
# ===================================================================
log.info("Etapa 1: preparando as inscrições do SIRF...")

# Inscrição do SIRF, sem espaços nas pontas.
inscricao_sirf = df_sirf[COL_SIRF_INSCRICAO].fillna("").str.strip()


def inscricao_area_maior(serie):
    """Troca o último bloco da inscrição por 0000 (a 'área maior' de um lote desdobrado)."""
    # Separa nos pontos: a inscrição tem 4 blocos, como "10.0006.0001.0000".
    partes = serie.str.split(".")
    # Só faz sentido quando há exatamente 4 blocos; os demais ficam vazios.
    tem_quatro = partes.str.len() == 4
    # Junta os 3 primeiros blocos e acrescenta ".0000".
    maior = partes.str[:3].str.join(".") + ".0000"
    return maior.where(tem_quatro)


inscricao_maior = inscricao_area_maior(inscricao_sirf)

log.info(f"Etapa 1 concluída! {len(inscricao_sirf):,} registros do SIRF.")

# ===================================================================
# ETAPA 2: PROCURAR O PONTO DE CADA REGISTRO
# ===================================================================
log.info("\nEtapa 2: procurando o ponto do lote de cada registro...")

# Dicionários "inscrição -> latitude" e "inscrição -> longitude", para a busca ser rápida.
lat_por_inscricao = dict(zip(df_lotes["inscricao"], df_lotes["lat"]))
lon_por_inscricao = dict(zip(df_lotes["inscricao"], df_lotes["lon"]))

# Primeiro tenta a inscrição exata.
lat_direto = inscricao_sirf.map(lat_por_inscricao)
lon_direto = inscricao_sirf.map(lon_por_inscricao)
achou_direto = lat_direto.notna()

# Para quem não achou, tenta a área maior.
lat_maior = inscricao_maior.map(lat_por_inscricao)
lon_maior = inscricao_maior.map(lon_por_inscricao)
achou_maior = ~achou_direto & lat_maior.notna()

# Junta os dois: o direto tem prioridade.
lat_final = lat_direto.where(achou_direto, lat_maior)
lon_final = lon_direto.where(achou_direto, lon_maior)
# Texto dizendo como o ponto foi obtido (serve para conferir depois).
origem_ponto = np.select([achou_direto, achou_maior], ["lote direto", "área maior"], default="sem lote")

log.info(f"Etapa 2 concluída! Lote direto: {int(achou_direto.sum()):,} | "
      f"área maior: {int(achou_maior.sum()):,} | sem lote: {int((~achou_direto & ~achou_maior).sum()):,}.")

# ===================================================================
# ETAPA 3: LIMPAR O ENDEREÇO
# ===================================================================
log.info("\nEtapa 3: limpando o endereço do SIRF...")

# Número do imóvel normalizado (primeiro grupo de dígitos).
log.info("    Normalizando o número do endereço...")
num_sirf = normalizar_numero(df_sirf[COL_SIRF_NUMERO])
# CEP sem espaços nas pontas.
cep_sirf = df_sirf[COL_SIRF_CEP].fillna("").str.strip()
# Logradouro em maiúsculas, sem acentos e sem pontuação.
log.info("    Normalizando o nome da via...")
via_sirf = normalizar_texto(df_sirf[COL_SIRF_LOGRADOURO])
# Troca a abreviação do tipo (R, AV, PCA...) pelo nome completo. O log lista todas as encontradas.
log.info("    Expandindo a abreviação do tipo da via...")
via_sirf = expandir_tipo_sirf(via_sirf, "SIRF")
# Tira do fim da via o número do próprio imóvel, quando ele aparece lá.
log.info("    Tirando o número do fim do nome da via...")
via_sirf = tirar_numero_do_fim(via_sirf, num_sirf)
# Aplica a limpeza completa (as mesmas regras usadas nas outras bases).
log.info("    Limpando o nome da via...")
via_sirf = limpar_via(via_sirf, "SIRF")

log.info("Etapa 3 concluída!")

# ===================================================================
# ETAPA 4: MONTAR O ref_pref
# ===================================================================
log.info("\nEtapa 4: montando a base de endereços da Prefeitura (ref_pref)...")

# Tabela com uma linha por registro do SIRF.
ref_pref = pd.DataFrame({
    "cep": cep_sirf.mask(cep_sirf == ""),          # CEP vazio vira "sem CEP" (NaN)
    "via": via_sirf.mask(via_sirf == ""),          # via vazia vira "sem via" (NaN)
    "num": num_sirf,                               # número já normalizado
    "lat": lat_final,
    "lon": lon_final,
    "origem_ponto": origem_ponto,
})

# Só entram os registros que têm ponto E têm via (sem via não dá para cruzar endereço).
sem_ponto = ref_pref["lat"].isna()
sem_via = ref_pref["via"].isna()
ref_pref = ref_pref[~sem_ponto & ~sem_via].reset_index(drop=True)

log.info(f"Etapa 4 concluída! ref_pref: {len(ref_pref):,} linhas "
      f"({int(sem_ponto.sum()):,} sem ponto e {int((~sem_ponto & sem_via).sum()):,} sem via ficaram de fora).")

# ===================================================================
# ETAPA 5: RESUMO
# ===================================================================
log.info("\nEtapa 5: calculando o resumo...")

log.info("\n--- RESUMO DA BASE DA PREFEITURA ---")
log.info(f"Linhas: {len(ref_pref):,}")
log.info(f"  com número: {int(ref_pref['num'].notna().sum()):,} | sem número: {int(ref_pref['num'].isna().sum()):,}")
log.info(f"  com CEP: {int(ref_pref['cep'].notna().sum()):,} | sem CEP: {int(ref_pref['cep'].isna().sum()):,}")
log.info(f"Vias distintas: {ref_pref['via'].nunique():,} | CEPs distintos: {ref_pref['cep'].nunique():,}")
log.info("Como o ponto foi obtido:")
for nome_origem, qtd in ref_pref["origem_ponto"].value_counts().items():
    log.info(f"  {nome_origem}: {qtd:,}")

# Quantos CEPs cada via tem (ajuda a entender se o SIRF usa um CEP por rua ou por trecho).
ceps_por_via_pref = ref_pref.dropna(subset=["cep"]).groupby("via")["cep"].nunique()
log.info("\nQuantidade de CEPs por via na base da Prefeitura:")
log.info(f"  vias com 1 CEP: {int((ceps_por_via_pref == 1).sum()):,}")
log.info(f"  vias com 2 a 5 CEPs: {int(((ceps_por_via_pref >= 2) & (ceps_por_via_pref <= 5)).sum()):,}")
log.info(f"  vias com mais de 5 CEPs: {int((ceps_por_via_pref > 5).sum()):,}")

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b11)}.")

## 2. Gravação e validação
Grava `ref_pref` e `sirf_vias` em Parquet, depois de conferir linhas e nulos.

In [ ]:
log.info("SEÇÃO 2: gravação e validação...")
# CEP e via de TODO o SIRF (com e sem ponto), na mesma ordem do df_sirf.
sirf_vias = pd.DataFrame({"cep": cep_sirf, "via": via_sirf}).reset_index(drop=True)
validacao.validar_tabela("ref_pref", ref_pref, obrigatorias=["via", "lat", "lon"])
validacao.validar_tabela("sirf_vias", sirf_vias, linhas_entrada=len(df_sirf))
assert len(sirf_vias) == len(df_sirf), "sirf_vias precisa ter uma linha por linha do df_sirf"
fontes.gravar_processamento(cfg, "ref_pref", ref_pref)
fontes.gravar_processamento(cfg, "sirf_vias", sirf_vias)
log.info("SEÇÃO 2 concluída!")

## 3. Resumo
Números principais, arquivos gerados e tempo total.

In [ ]:
log.info("RESUMO do 04_base_enderecos_prefeitura...")
log.info(f"ref_pref: {len(ref_pref):,} endereços com ponto | vias distintas: {ref_pref['via'].nunique():,}")
log.info("Arquivos gerados:")
for chave in ['ref_pref', 'sirf_vias']:
    caminho = cfg.caminho("processamento." + chave)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")